In [ ]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import pandas as pd
from utils.preprocess import *
import os
import sys
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from torchcfm.utils import plot_trajectories, torch_wrapper
from simulate.simulate import *
from omegaconf import OmegaConf
from utils.hydra import *
from datasets.process import *
from scripts.run_model import *
from eval.eval import *
import matplotlib.pyplot as plt
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
############################################################

In [ ]:
config = load_config(overrides=['dataset=zebrafish'])
OmegaConf.set_struct(config, False)

# Override for mouse
config.project = "mouse"
config.dataset = "mouse"
config.pc_dim = 100
config.t0_index = 0   # E6.5
config.t1_index = 4   # E7.5

config.classifier.num_layers = 2
config.classifier.hidden_dim = 256
config.classifier.epsilon = 0.05
config.use_paga = True

In [ ]:
### SETTINGS ###

adata = process_data(pc_dim=config.pc_dim, data=config.dataset, use_paga=config.use_paga)

print(adata.obs['cell_type'].nunique())

timepoints = sorted(adata.obs['timepoint'].unique().tolist())
tree = adata.uns['tree']

config.num_classes = adata.obs['cell_type'].nunique()

print(f"Timepoints: {timepoints}")
print(f"t0_index: {config.t0_index}, t1_index: {config.t1_index}")
print(f"t0: {timepoints[config.t0_index]}, t1: {timepoints[config.t1_index]}")
print(f"Shape: {adata.shape}")

In [ ]:
# Explore cell types and stages
print("Cell types:")
for ct in sorted(adata.obs['cell_type'].unique()):
    n = (adata.obs['cell_type'] == ct).sum()
    print(f"  {ct}: {n}")

print(f"\nStages:")
for t in timepoints:
    n = (adata.obs['timepoint'] == t).sum()
    print(f"  {t}: {n}")

In [ ]:
# # Visualize tree structure
# itos = adata.uns['itos']
# N = adata.uns['tree'].shape[0]
# print("Tree edges:")
# for i in range(N):
#     for j in range(N):
#         if adata.uns['tree'][i][j] == 1.0:
#             print(f"  {itos[i]} -> {itos[j]}")

In [ ]:
# #CFM
# config.metric = "cfm"
# config.no_learning = True
# config.metric_max_epochs = 2

In [ ]:
# #CFM + UOT
# config.metric = "cfm"
# config.no_learning = True

# config.method = "unbalanced"
# config.reg = 0.01
# config.reg_m = 1.0

In [ ]:
# #CFM + Finsler
# config.metric = "cfm"

# config.finsler.use = True
# config.finsler.lamb = 0.5
# config.balance_classes = False
# config.metric_max_epochs = 2

In [ ]:
# #MFM-Euc
# config.metric = "mfm"
# config.mfm.use_euclidean_ot = True

# config.mfm.K = 150
# config.mfm.kappa = 1.5
# config.mfm.epsilon = 1e-1

In [ ]:
#MFM
config.metric = "mfm"

config.mfm.K = 150
config.mfm.kappa = 1.5
config.mfm.epsilon = 1e-1

In [ ]:
#MFM + Finsler
config.metric = "mfm"

config.mfm.K = 150
config.mfm.kappa = 1.5
config.mfm.epsilon = 1e-1

# config.finsler.use = True
# config.finsler.lamb = 0.5

In [ ]:
adata

In [ ]:
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)

In [ ]:
# fig, axs = plt.subplots(len(timepoints), figsize=(12, 3*len(timepoints)))
# for i, t in enumerate(timepoints):
#     sc.pl.umap(adata[adata.obs['timepoint'] == t], color='cell_type', ax=axs[i], title=f"time {t}", show=False)
# plt.tight_layout()
# plt.show()

In [ ]:
t0, t1 = timepoints[config.t0_index], timepoints[config.t1_index]
adata = adata[(adata.obs['timepoint'] >= t0) & (adata.obs['timepoint'] <= t1)]
adata_train = adata[adata.obs['timepoint'].isin([t0, t1])]
print(t0, t1)

In [ ]:
singleton_dataloader = build_singleton_dataloader(config, adata_train)
paired_dataloader = build_paired_dataloader(config, adata_train)

In [ ]:
classifier_model, metric_model, embed_model, flow_model = run_full_model(config=config,
                                                                         project=config.project,
                                                                         singleton_dataloader=singleton_dataloader,
                                                                         paired_dataloader=paired_dataloader,
                                                                         timepoints=timepoints,
                                                                         tree=tree)

In [ ]:
remove_all_forward_hooks(classifier_model)
remove_all_forward_hooks(metric_model)
remove_all_forward_hooks(embed_model)
remove_all_forward_hooks(flow_model)

In [ ]:
t0, t1 = timepoints[config.t0_index], timepoints[config.t1_index]

w1_scores = []
for index in range(config.t0_index + 1, config.t1_index):
    print(index)
    t = timepoints[index]
    w1 = predict(embed_model, adata, t0, t, t1, num_traj=6000, library="pot")
    w1_scores.append(w1)
w1_scores = torch.tensor(w1_scores)
print(w1_scores)
print(torch.mean(w1_scores))

In [ ]:
device = embed_model.device

batch = next(iter(paired_dataloader))
x0, x1, _, _ = embed_model._prepare_batch(batch)

paths = embed_model.sample_geodesic_path(batch, num_points=50)
paths_flat = paths.reshape(-1, paths.shape[-1])

time_axis = np.linspace(0, 1, 50)

with torch.no_grad():
    logits = classifier_model.classify(torch.tensor(paths_flat, device=device).float())
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    
probs = probs.reshape(paths.shape[0], paths.shape[1], -1)

itos = adata.uns['itos']
cell_types = [itos[i] for i in range(len(itos))] + ["outlier"]

fig, axes = plt.subplots(5, figsize=(12, 8), sharex=True, sharey=True)

flat_axes = axes.flatten()

for i in range(len(flat_axes)):
    traj = probs[:,i]
    ax = flat_axes[i]
    ax.stackplot(time_axis, traj.T, labels=cell_types, alpha=0.8, colors=plt.cm.tab20.colors)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

handles, labels = flat_axes[0].get_legend_handles_labels()

fig.legend(
    handles, 
    labels, 
    loc='center right',
    bbox_to_anchor=(1, 0.5),
    title="Predicted Cell Type"
)

plt.tight_layout(rect=[0, 0, 0.88, 1]) 
plt.show()

In [ ]:
start = paths[0,:30]
end = paths[-1,:30]
itos = adata.uns['itos']
start_classes = torch.argmax(classifier_model.classify(start), dim=1).detach().cpu().numpy()
start_classes = [itos[i] for i in start_classes]
end_classes = torch.argmax(classifier_model.classify(end), dim=1).detach().cpu().numpy()
end_classes = [itos[i] for i in end_classes]

for i in range(5):
    print(f"{start_classes[i]:<50} {end_classes[i]:>50}")

In [ ]:
#EMBED VALIDATION
if config.metric == "cfm" and not config.finsler.use:
    x0, x1, _, _= embed_model._prepare_batch(batch)
    x0, x1 = x0[0], x1[0]
    x0_, x1_, _, _ = embed_model.flow_matcher.ot_sampler.sample_plan(x0, x1)
    true_dist = torch.norm(x0 - x1, dim = -1)
    f = embed_model.embed_fn
    learned_dist = torch.norm(f(x0).detach() - f(x1).detach(), dim=-1)
    print(torch.stack([true_dist, learned_dist], dim=1)[:10])

In [ ]:
paths.shape

In [ ]:
# # PHATE Visualization: Ground Truth, Predictions, Entropy, Trajectories

# import phate
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.neighbors import NearestNeighbors
# from mpl_toolkits.axes_grid1 import make_axes_locatable

# def project_to_phate(new_points, X_original, X_phate_embedding, k=10):
#     """Project points to PHATE space via KNN interpolation."""
#     knn = NearestNeighbors(n_neighbors=k, metric='euclidean')
#     knn.fit(X_original)
#     distances, indices = knn.kneighbors(new_points)
#     weights = 1.0 / (distances + 1e-10)
#     weights = weights / weights.sum(axis=1, keepdims=True)
#     projected = np.zeros((len(new_points), 2))
#     for i in range(len(new_points)):
#         projected[i] = np.average(X_phate_embedding[indices[i]], axis=0, weights=weights[i])
#     return projected

# # Fit PHATE on PCA data
# X_data = adata.obsm['X_pca']
# phate_op = phate.PHATE(n_components=2, knn=15, t='auto', random_state=42, verbose=True)
# X_phate = phate_op.fit_transform(X_data)
# adata.obsm['X_phate'] = X_phate

# # Project trajectories to PHATE space
# paths_flat_np = paths.reshape(-1, paths.shape[-1])
# if isinstance(paths_flat_np, torch.Tensor):
#     paths_flat_np = paths_flat_np.detach().cpu().numpy()
# traj_phate = project_to_phate(paths_flat_np, X_data, X_phate, k=10)
# traj_phate = traj_phate.reshape(paths.shape[0], paths.shape[1], 2)

# # Compute classifier predictions and entropy
# device = classifier_model.device
# with torch.no_grad():
#     X_tensor = torch.tensor(X_data, device=device, dtype=torch.float32)
#     logits = classifier_model.classify(X_tensor)
#     probs = torch.softmax(logits, dim=1).cpu().numpy()

# ground_truth = adata.obs['cell_type'].values
# predicted_classes = np.argmax(probs, axis=1)
# itos = adata.uns['itos']
# predicted_labels = np.array([itos[i] if i < len(itos) else 'outlier' for i in predicted_classes])

# epsilon = 1e-10
# entropy = -np.sum(probs * np.log(probs + epsilon), axis=1)
# entropy_normalized = entropy / np.log(probs.shape[1])

# # Create 2x2 figure
# fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# unique_cell_types = adata.obs['cell_type'].unique()
# palette = sns.color_palette('tab20', len(unique_cell_types))
# color_map = {ct: palette[i] for i, ct in enumerate(unique_cell_types)}

# # Panel 1: Ground Truth Labels
# ax1 = axs[0, 0]
# for ct in unique_cell_types:
#     mask = adata.obs['cell_type'] == ct
#     ax1.scatter(X_phate[mask, 0], X_phate[mask, 1], c=[color_map[ct]], label=ct, s=5, alpha=0.6)
# ax1.set_title('Ground Truth Cell Types')
# ax1.set_xlabel('PHATE 1')
# ax1.set_ylabel('PHATE 2')

# # Panel 2: Classifier Predicted Labels
# ax2 = axs[0, 1]
# for ct in unique_cell_types:
#     mask = predicted_labels == ct
#     if mask.sum() > 0:
#         ax2.scatter(X_phate[mask, 0], X_phate[mask, 1], c=[color_map[ct]], label=ct, s=5, alpha=0.6)
# outlier_mask = predicted_labels == 'outlier'
# if outlier_mask.sum() > 0:
#     ax2.scatter(X_phate[outlier_mask, 0], X_phate[outlier_mask, 1], c='gray', label='outlier', s=5, alpha=0.6)
# ax2.set_title('Classifier Predicted Cell Types')
# ax2.set_xlabel('PHATE 1')
# ax2.set_ylabel('PHATE 2')

# # Panel 3: Classifier Entropy
# ax3 = axs[1, 0]
# scatter3 = ax3.scatter(X_phate[:, 0], X_phate[:, 1], c=entropy_normalized, cmap='plasma', s=5, alpha=0.7, vmin=0, vmax=1)
# ax3.set_title('Classifier Entropy (Uncertainty)')
# ax3.set_xlabel('PHATE 1')
# ax3.set_ylabel('PHATE 2')
# divider3 = make_axes_locatable(ax3)
# cax3 = divider3.append_axes("right", size="5%", pad=0.05)
# plt.colorbar(scatter3, cax=cax3, label='Normalized Entropy')

# # Panel 4: Trajectories
# ax4 = axs[1, 1]
# ax4.scatter(X_phate[:, 0], X_phate[:, 1], c='lightgray', s=3, alpha=0.3)
# num_traj = 5
# traj_indices = np.linspace(0, paths.shape[1]-1, num_traj, dtype=int)
# colors = plt.cm.viridis(np.linspace(0, 1, num_traj))
# for idx, ti in enumerate(traj_indices):
#     traj = traj_phate[:, ti, :]
#     ax4.plot(traj[:, 0], traj[:, 1], c=colors[idx], linewidth=2, alpha=0.8)
#     ax4.scatter(traj[0, 0], traj[0, 1], c='green', s=50, marker='o', edgecolor='black', zorder=5)
#     ax4.scatter(traj[-1, 0], traj[-1, 1], c='red', s=50, marker='s', edgecolor='black', zorder=5)
# ax4.set_title('Learned Geodesic Trajectories')
# ax4.set_xlabel('PHATE 1')
# ax4.set_ylabel('PHATE 2')

# handles, labels = ax1.get_legend_handles_labels()
# fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.15, 0.5), title='Cell Types', fontsize=8)
# plt.tight_layout(rect=[0, 0, 0.88, 1])
# plt.suptitle('PHATE Visualization of Mouse Gastrulation', fontsize=14, y=1.02)
# plt.show()